# Extractive QA — Colab (fases 3, 4 y 5)
Este notebook solo orquesta: el trabajo pesado está en `train.py` y `evaluate.py`.

**Antes de empezar:** sube `train.py`, `evaluate.py` (y `public_sample.json` cuando lo tengas) a `MyDrive/ExtractiveQA/src/`
y elige un runtime con GPU (*Runtime → Change runtime type*).

## Fase 3 — Montar Drive y verificar archivos

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
DRIVE_ROOT = Path("/content/drive/MyDrive/ExtractiveQA")
PILOT_ROOT = Path("/content/drive/MyDrive/ExtractiveQA_pilot")   # separado a propósito (ver fase 4)
SRC_DIR    = DRIVE_ROOT / "src"

for f in ("train.py", "evaluate.py"):
    assert (SRC_DIR / f).exists(), f"Falta {SRC_DIR / f}: súbelo a Drive"
!nvidia-smi -L

In [ ]:
# Colab ya trae torch: NO lo reinstales.
!pip -q install -U transformers datasets accelerate

## Fase 4 — Entrenamiento

**Piloto** (mide tiempo por época y memoria). Usa su propia carpeta para que sus checkpoints
no se confundan con los de la corrida completa: el script reanuda automáticamente desde el último
checkpoint que encuentre en `checkpoints/`.

In [ ]:
!python -u "{SRC_DIR}/train.py" --drive_root "{PILOT_ROOT}" --train_samples 25000 --epochs 2

**Corrida completa.** Si Colab se desconecta: vuelve a ejecutar la celda de montaje (fase 3) y esta misma celda.
Reanuda desde el último checkpoint completo, con optimizador y scheduler.

In [ ]:
!python -u "{SRC_DIR}/train.py" --drive_root "{DRIVE_ROOT}" --epochs 3

## Curvas de loss (entregable)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

LOG = DRIVE_ROOT / "logs"
tr = pd.read_csv(LOG / "train_metrics.csv").drop_duplicates("step", keep="last")
ev = pd.read_csv(LOG / "eval_metrics.csv").drop_duplicates("step", keep="last")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(tr["step"], tr["loss"], alpha=0.7, label="train loss")
ax.plot(ev["step"], ev["eval_loss"], "o-", label="eval loss")
ax.set_xlabel("step"); ax.set_ylabel("loss"); ax.legend(); ax.grid(alpha=0.3)
fig.savefig(LOG / "loss_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## Fase 5 — Evaluación y calibración de `null`
Recorre la validación completa de SQuAD 2.0, barre umbrales, elige el de mayor F1, verifica que cada respuesta
sea un fragmento exacto y corre `public_sample.json` si existe.

In [ ]:
sample = SRC_DIR / "public_sample.json"
extra = f'--public_sample "{sample}"' if sample.exists() else ""
!python -u "{SRC_DIR}/evaluate.py" --drive_root "{DRIVE_ROOT}" {extra}

In [ ]:
import json
RES = DRIVE_ROOT / "results"

summary = json.loads((RES / "eval_summary.json").read_text())
print("delta = 0     :", {k: round(v, 2) for k, v in summary["at_delta_0"].items()})
print("mejor umbral  :", {k: round(v, 2) for k, v in summary["at_best_threshold"].items()})
print("respuestas que no son fragmento exacto:", summary["span_violations"])

sw = pd.read_csv(RES / "threshold_sweep.csv")
best = sw.loc[sw["f1"].idxmax()]
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(sw["threshold"], sw["f1"], label="F1")
ax.plot(sw["threshold"], sw["has_ans_f1"], label="HasAns F1", alpha=0.6)
ax.plot(sw["threshold"], sw["no_ans_acc"], label="NoAns acc", alpha=0.6)
ax.axvline(best["threshold"], color="k", ls="--", label=f"mejor t={best['threshold']:.2f}")
ax.set_xlabel("umbral de delta"); ax.set_ylabel("%"); ax.legend(); ax.grid(alpha=0.3)
fig.savefig(RES / "threshold_sweep.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Resultados de la muestra pública y archivos que viajan al despliegue local
if (RES / "public_sample_results.json").exists():
    print(json.dumps(json.loads((RES / "public_sample_results.json").read_text()), indent=2, ensure_ascii=False))

print(*sorted(p.name for p in (DRIVE_ROOT / "final_model").iterdir()), sep="\n")
print(json.dumps(json.loads((DRIVE_ROOT / "final_model" / "postprocess_config.json").read_text()), indent=2))